# Binary Search Trees (BST) & Binary Trees in Python

> **Topic:** Binary Search Trees & Tree Algorithms | **Folder:** Data Structures & Algorithms

A **Binary Tree** is a hierarchical data structure where each node has at most two children (`left` and `right`).
A **Binary Search Tree (BST)** is a specialized binary tree that satisfies the **BST Invariant**:
- For any node $X$, all keys in its **left subtree** are strictly smaller than $X$ ($Left < X$).
- For any node $X$, all keys in its **right subtree** are strictly greater than $X$ ($Right > X$).

---

## Table of Contents
1. [Tree Terminology & BST Invariant](#1.-Tree-Terminology-&-BST-Invariant)
2. [Time & Space Complexity of BST Operations](#2.-Time-&-Space-Complexity-of-BST-Operations)
3. [Building a Custom BST Class from Scratch](#3.-Building-a-Custom-BST-Class-from-Scratch)
4. [Tree Traversal Algorithms (DFS: In-Order, Pre-Order, Post-Order & BFS Level-Order)](#4.-Tree-Traversal-Algorithms-(DFS:-In-Order,-Pre-Order,-Post-Order-&-BFS-Level-Order))
5. [Algorithmic Pattern 1: Validate Binary Search Tree](#5.-Algorithmic-Pattern-1:-Validate-Binary-Search-Tree)
6. [Algorithmic Pattern 2: Lowest Common Ancestor (LCA) in BST](#6.-Algorithmic-Pattern-2:-Lowest-Common-Ancestor-(LCA)-in-BST)
7. [Algorithmic Pattern 3: Convert Sorted Array to Balanced BST](#7.-Algorithmic-Pattern-3:-Convert-Sorted-Array-to-Balanced-BST)
8. [Algorithmic Pattern 4: Kth Smallest Element in a BST](#8.-Algorithmic-Pattern-4:-Kth-Smallest-Element-in-a-BST)
9. [Algorithmic Pattern 5: Serialize & Deserialize Binary Tree](#9.-Algorithmic-Pattern-5:-Serialize-&-Deserialize-Binary-Tree)
10. [Quick Reference Card](#10.-Quick-Reference-Card)


---
## 1. Tree Terminology & BST Invariant

- **Root**: The topmost node of the tree.
- **Leaf**: A node with no children (`left == None` and `right == None`).
- **Height ($h$)**: The number of edges on the longest path from root to leaf.
- **BST Invariant**: `node.left.val < node.val < node.right.val` (for all nodes).


In [ ]:
# Basic TreeNode structure
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"

# Constructing a simple BST manually:
#        10
#       /  \
#      5    15
root = TreeNode(10)
root.left = TreeNode(5)
root.right = TreeNode(15)

print("Root        :", root.val)
print("Left Child  :", root.left.val)
print("Right Child :", root.right.val)


---
## 2. Time & Space Complexity of BST Operations

| Operation | Average Case (Balanced Tree) | Worst Case (Skewed Tree) | Space Complexity |
|-----------|------------------------------|--------------------------|------------------|
| **Search `search(val)`** | $O(\log n)$ | $O(n)$ | $O(h)$ recursion stack |
| **Insertion `insert(val)`** | $O(\log n)$ | $O(n)$ | $O(h)$ |
| **Deletion `delete(val)`** | $O(\log n)$ | $O(n)$ | $O(h)$ |
| **Traversals** | $O(n)$ | $O(n)$ | $O(h)$ |

> **Note**: Self-balancing BSTs (AVL, Red-Black Trees) guarantee $O(\log n)$ worst-case height.


---
## 3. Building a Custom BST Class from Scratch

Handles insertion, searching, and 3-case node deletion:
1. **Leaf node**: Simply remove.
2. **Single child**: Bypass node to its child.
3. **Two children**: Replace with **In-Order Successor** (smallest node in right subtree), then delete successor.


In [ ]:
class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, val):
        self.root = self._insert(self.root, val)

    def _insert(self, node, val):
        if not node: return TreeNode(val)
        if val < node.val: node.left = self._insert(node.left, val)
        elif val > node.val: node.right = self._insert(node.right, val)
        return node

    def search(self, val):
        curr = self.root
        while curr:
            if val == curr.val: return True
            elif val < curr.val: curr = curr.left
            else: curr = curr.right
        return False

    def delete(self, val):
        self.root = self._delete(self.root, val)

    def _delete(self, node, val):
        if not node: return None
        if val < node.val: node.left = self._delete(node.left, val)
        elif val > node.val: node.right = self._delete(node.right, val)
        else:
            # Case 1 & 2: 0 or 1 child
            if not node.left: return node.right
            if not node.right: return node.left
            # Case 3: 2 children -> Find in-order successor (min in right subtree)
            succ = self._min_node(node.right)
            node.val = succ.val
            node.right = self._delete(node.right, succ.val)
        return node

    def _min_node(self, node):
        curr = node
        while curr.left: curr = curr.left
        return curr

# Testing BST
bst = BinarySearchTree()
for x in [50, 30, 70, 20, 40, 60, 80]: bst.insert(x)
print("Search 40:", bst.search(40))  # True
print("Search 90:", bst.search(90))  # False
bst.delete(30)                        # Delete node with 2 children
print("Search 30 after deletion:", bst.search(30))  # False


---
## 4. Tree Traversal Algorithms

| Traversal | Order | Characteristic | Key Application |
|-----------|-------|----------------|-----------------|
| **In-Order (DFS)** | Left $\rightarrow$ Root $\rightarrow$ Right | Yields values in **sorted order** | Sorting, BST verification |
| **Pre-Order (DFS)** | Root $\rightarrow$ Left $\rightarrow$ Right | Visits root before subtrees | Tree copying, Serialization |
| **Post-Order (DFS)** | Left $\rightarrow$ Right $\rightarrow$ Root | Visits children before root | Deleting tree, Subtree evaluation |
| **Level-Order (BFS)** | Level-by-level (Top to Bottom) | Queue-based BFS | Level printing, shortest path |


In [ ]:
from collections import deque

def in_order_traversal(root):
    res = []
    def dfs(node):
        if not node: return
        dfs(node.left)
        res.append(node.val)
        dfs(node.right)
    dfs(root)
    return res

def pre_order_traversal(root):
    res = []
    def dfs(node):
        if not node: return
        res.append(node.val)
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return res

def post_order_traversal(root):
    res = []
    def dfs(node):
        if not node: return
        dfs(node.left)
        dfs(node.right)
        res.append(node.val)
    dfs(root)
    return res

# Build sample tree: root = bst.root
sample_bst = BinarySearchTree()
for x in [4, 2, 6, 1, 3, 5, 7]: sample_bst.insert(x)
r = sample_bst.root

print("In-Order   (Sorted):", in_order_traversal(r))
print("Pre-Order          :", pre_order_traversal(r))
print("Post-Order         :", post_order_traversal(r))


---
## 5. Algorithmic Pattern 1: Validate Binary Search Tree

Validates if a tree satisfies the BST property using min/max range bounds check in **$O(n)$ time**.


In [ ]:
def is_valid_bst(root):
    def validate(node, low=float('-inf'), high=float('inf')):
        if not node: return True
        if not (low < node.val < high): return False
        return validate(node.left, low, node.val) and validate(node.right, node.val, high)
    return validate(root)

# Valid BST
valid_root = TreeNode(10, TreeNode(5), TreeNode(15))
print("Is valid BST?", is_valid_bst(valid_root))

# Invalid BST (12 is in left subtree of 10!)
invalid_root = TreeNode(10, TreeNode(12), TreeNode(15))
print("Is invalid BST valid?", is_valid_bst(invalid_root))


---
## 6. Algorithmic Pattern 2: Lowest Common Ancestor (LCA) in BST

Exploits BST properties: if both target nodes $P$ and $Q$ are smaller than `root`, go left.  
If both are greater, go right. Otherwise, `root` is the split point (LCA) in **$O(h)$ time** and **$O(1)$ space**.


In [ ]:
def lowest_common_ancestor(root, p, q):
    curr = root
    while curr:
        if p.val < curr.val and q.val < curr.val:
            curr = curr.left
        elif p.val > curr.val and q.val > curr.val:
            curr = curr.right
        else:
            return curr  # Found split node (LCA)
    return None

# Tree: 6 -> (2 -> (0, 4), 8)
n6 = TreeNode(6); n2 = TreeNode(2); n8 = TreeNode(8)
n0 = TreeNode(0); n4 = TreeNode(4)
n6.left = n2; n6.right = n8; n2.left = n0; n2.right = n4

print("LCA of 0 and 4:", lowest_common_ancestor(n6, n0, n4).val)  # Returns 2
print("LCA of 2 and 8:", lowest_common_ancestor(n6, n2, n8).val)  # Returns 6


---
## 7. Algorithmic Pattern 3: Convert Sorted Array to Balanced BST

Converts a sorted array to a height-balanced BST in **$O(n)$ time** using divide-and-conquer (middle element as root).


In [ ]:
def sorted_array_to_bst(nums):
    if not nums: return None
    mid = len(nums) // 2
    root = TreeNode(nums[mid])
    root.left = sorted_array_to_bst(nums[:mid])
    root.right = sorted_array_to_bst(nums[mid + 1:])
    return root

sorted_nums = [-10, -3, 0, 5, 9]
balanced_root = sorted_array_to_bst(sorted_nums)
print("In-Order of reconstructed BST:", in_order_traversal(balanced_root))


---
## 8. Algorithmic Pattern 4: Kth Smallest Element in a BST

Uses iterative in-order traversal with a stack to find the $K$-th smallest element in **$O(h + k)$ time**.


In [ ]:
def kth_smallest(root, k):
    stack = []
    curr = root
    while curr or stack:
        while curr:
            stack.append(curr)
            curr = curr.left
        curr = stack.pop()
        k -= 1
        if k == 0: return curr.val
        curr = curr.right
    return -1

print("2nd smallest in sample_bst:", kth_smallest(sample_bst.root, 2))


---
## 9. Algorithmic Pattern 5: Serialize & Deserialize Binary Tree


In [ ]:
class Codec:
    def serialize(self, root):
        vals = []
        def dfs(node):
            if not node:
                vals.append("#")
                return
            vals.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return ",".join(vals)

    def deserialize(self, data):
        vals = iter(data.split(","))
        def dfs():
            val = next(vals)
            if val == "#": return None
            node = TreeNode(int(val))
            node.left = dfs()
            node.right = dfs()
            return node
        return dfs()

codec = Codec()
serialized_str = codec.serialize(sample_bst.root)
print("Serialized Tree String:", serialized_str)
deserialized_root = codec.deserialize(serialized_str)
print("Deserialized In-Order :", in_order_traversal(deserialized_root))


---
## 10. Quick Reference Card


In [ ]:
# ==================================================================
# BINARY SEARCH TREES – QUICK REFERENCE
# ==================================================================

# In-Order Traversal (Yields Sorted Array):
# def in_order(node):
#     if node: yield from in_order(node.left); yield node.val; yield from in_order(node.right)

# Validate BST Range Check:
# def isValid(n, lo=float('-inf'), hi=float('inf')):
#     return not n or (lo < n.val < hi and isValid(n.left, lo, n.val) and isValid(n.right, n.val, hi))


---
## Summary

| Operation / Algorithm | Average Complexity | Key Property |
|-----------------------|--------------------|--------------|
| **In-Order Traversal** | $O(n)$ | Traverses nodes in strictly sorted order |
| **BST Validation** | $O(n)$ | Range bounds verification ($low < val < high$) |
| **BST LCA** | $O(h)$ time, $O(1)$ space | Split point node traversal |
| **Sorted Array to BST** | $O(n)$ | Middle element as root recursive allocation |
| **Serialization** | $O(n)$ | Pre-order traversal with null marker encoding |

---
*Next up: **Graph Algorithms***
